In [ ]:
import pandas as pd

# csv als DataFrame einlesen
df = pd.read_csv("project/survey_results_public.csv", index_col="ResponseId")

#Spalten mit SO entfernen
df = df.drop(columns=df.columns[df.columns.str.startswith('SO')])

#EdLevel splitten um nur EdLevel anzuzeigen
df["EdLevel"] = df["EdLevel"].str.split("(").str[0].str.strip()

#RemoteWork auf numerische Werte mappen
remote_map = {
    "Remote": 0,
    "In-person": 1,
    "Hybrid (some remote, leans heavy to in-person)": 0.75,
    "Hybrid (some in-person, leans heavy to flexibility)": 0.25,
    "Your choice (very flexible, you can come in when you want or just as needed)": 0.5
}
df.insert(
    df.columns.get_loc("RemoteWork") + 1,
    "RemoteCategoryNum",
    df["RemoteWork"].map(remote_map)
)

#Age zu numerischen Werten mappen, immer Mittelwert der ranges
age_map = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29,
    "35-44 years old": 39,
    "45-54 years old": 49,
    "55-64 years old": 59,
    "65 years or older": 70
}

df.insert(
    df.columns.get_loc("Age") + 1,
    "AgeNum",
    df['Age'].map(age_map)
)

#Über 70 jährige entfernen
df = df[df['AgeNum'] <= 65]


df.to_csv("survey_results_shortened.csv", index=False)


# 🧹 Datenbereinigung & Vereinheitlichung – To-Do Liste

Diese To-Do-Liste beschreibt alle notwendigen Schritte, um die Survey-CSV-Datei zu bereinigen, zu vereinheitlichen und für spätere Analysen oder Visualisierungen nutzbar zu machen.

---

## 1. Fehlende Werte standardisieren

### Kategoriale Spalten
- `NaN` → **"Keine Angabe"**
- Datentyp `object` beibehalten

### Numerische Spalten
- `NaN` **nicht ersetzen**
- Datentyp `int`/`float` belassen
  → wichtig für statistische Auswertungen (Durchschnitt, Median, Histogramme)

In [ ]:
import unicodedata
import re

In [ ]:
# Alle Spalten mit numerischen Werten in einen DataFrame packen
numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Alle Spalten mit nicht numerischen Werten in einen DataFrame packen
category_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Anzahl numerischer Spalten:", len(numeric_columns)) #Anzahl: 42
print("Anzahl kategorischer Spalten:", len(category_columns)) #Anzahl: 106

print("\nBeispiele numerischer Spalten:", numeric_columns[:10])
print("\nBeispiele kategorischer Spalten:", category_columns[:10])

## 2. Kategorische Text-Spalten bereinigen

### Ziel
Eine einheitliche und konsistente Darstellung, um spätere Gruppierungen und Analysen zu erleichtern.

### Maßnahmen
- Whitespace entfernen (Trimmen)
- Einheitliche Groß-/Kleinschreibung (z. B. `title()` oder `lower()`)
- Zusammenführen identischer kategorischer Werte mit unterschiedlicher Schreibweise
  *Beispiel: „Self taught“ und „self-taught“*
- Optional: Seltene Kategorien in **"Other"** gruppieren

### Beispiele betroffener Spalten
- `MainBranch`
- `EdLevel`
- `Employment`
- `Country`
- `OrgSize`
- `Industry`
- `AISelect`
- `AIPrimaryUse`

In [ ]:
# Normalisierungs-/Vereinheitlichungsfunktion für Strings zur Anwendung in nicht numerischen Spalten
def clean_text(s):
    if pd.isna(s):
        return s
    s = str(s).strip() # Leerzeichen am Anfang und Ende weg
    s = unicodedata.normalize("NFC", s) # Unicode normalisieren
    s = s.replace("–", "-").replace("—", "-")
    s = s.replace("’", "'") # Apostrophen vereinheitlichen
    s = re.sub(r"\s+", " ", s)
    return s

In [ ]:
# Alles in der Spalte zu lowercase ändern um besser damit zu arbeiten
def to_lowercase(s):
    if pd.isna(s):
        return s
    return s.lower()

In [ ]:
# Anwendung der Normalisierungs-/Vereinheitlichungsfunktion auf alle nicht numerischen Spalten
for col in category_columns:
    df[col] = df[col].apply(clean_text)

In [ ]:
# Currency außen vor wegen den Währungscodes wie USD oder EUR
exclude_columns = ["Country", "Currency"]

for col in category_columns:
    if col not in exclude_columns:
        df[col] = df[col].apply(to_lowercase)

## 3. Mehrfachauswahl-Spalten vereinheitlichen (`;`-getrennte Werte)

### Typische Probleme
- Uneinheitliche Formatierungen
- Semikolon-separierte Werte
- NaN-Werte
- Inkonsistente Reihenfolgen

### Maßnahmen
- Aufsplitten in Listen
  `"Python; JavaScript"` → `["Python", "JavaScript"]`
- Werte trimmen
- Duplikate in Listen entfernen
- Optionale alphabetische Sortierung der Werte
- NaN → **leere Liste** oder **"Keine Angabe"**

### Beispiele
- `DevType`
- `LanguageHaveWorkedWith`
- `LanguageWantToWorkWith`
- `DatabaseHaveWorkedWith`
- `ToolsTechHaveWorkedWith`
- `PlatformHaveWorkedWith`

In [ ]:
# Ausgabe aller Spalten, die Semikolon-getrennt sind
[col for col in df.columns if df[col].astype(str).str.contains(";").any()]

In [ ]:
# Alle Semikolon-getrennten Spalten in einer Liste
multi_select_cols = [
 'EmploymentAddl','LearnCode','AILearnHow','TechEndorse_13_TEXT','TechOppose_15_TEXT',
 'JobSatPoints_15_TEXT','LanguageHaveWorkedWith','LanguageWantToWorkWith','LanguageAdmired',
 'LanguagesHaveEntry','LanguagesWantEntry','DatabaseHaveWorkedWith','DatabaseWantToWorkWith',
 'DatabaseAdmired','DatabaseHaveEntry','DatabaseWantEntry','PlatformHaveWorkedWith',
 'PlatformWantToWorkWith','PlatformAdmired','PlatformWantEntry','WebframeHaveWorkedWith',
 'WebframeWantToWorkWith','WebframeAdmired','WebframeHaveEntry','WebframeWantEntry',
 'DevEnvsHaveWorkedWith','DevEnvsWantToWorkWith','DevEnvsAdmired','DevEnvHaveEntry',
 'DevEnvWantEntry','OpSysPersonal use','OpSysProfessional use',
 'OfficeStackAsyncHaveWorkedWith','OfficeStackAsyncWantToWorkWith','OfficeStackAsyncAdmired',
 'OfficeStackHaveEntry','CommPlatformHaveWorkedWith','CommPlatformWantToWorkWith',
 'CommPlatformAdmired','CommPlatformHaveEntr','CommPlatformWantEntr',
 'AIModelsHaveWorkedWith','AIModelsWantToWorkWith','AIModelsAdmired',
 'AIToolCurrently partially AI',"AIToolDon't plan to use AI for this task",
 'AIToolPlan to partially use AI','AIToolPlan to mostly use AI','AIToolCurrently mostly AI',
 'AIFrustration','AIExplain','AIAgent_Uses','AgentUsesGeneral',
 'AIAgentImpactSomewhat agree','AIAgentImpactNeutral','AIAgentImpactSomewhat disagree',
 'AIAgentImpactStrongly agree','AIAgentImpactStrongly disagree',
 'AIAgentChallengesNeutral','AIAgentChallengesSomewhat disagree',
 'AIAgentChallengesStrongly agree','AIAgentChallengesSomewhat agree',
 'AIAgentChallengesStrongly disagree','AIAgentKnowledge','AIAgentKnowWrite',
 'AIAgentOrchestration','AIAgentOrchWrite','AIAgentObserveSecure','AIAgentObsWrite',
 'AIAgentExternal','AIAgentExtWrite','AIHuman','AIOpen'
]

In [ ]:
def clean_multi_select(value):
    if pd.isna(value):
        return []

    splitted = str(value).split(";")

    cleaned = []
    for p in splitted:
        clean_text(p)
        if p:
            cleaned.append(p)

    cleaned = list(set(cleaned))

    cleaned.sort()

    return cleaned

In [ ]:
for col in multi_select_cols:
    df[col] = df[col].apply(clean_multi_select)

In [31]:
df

,MainBranch,Age,AgeNum,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
ResponseId,,,,,,,,,,,,,,,,,,,,,
1,i am a developer by profession,25-34 years old,29.0,master's degree,employed,"[caring for dependents (children, elderly, etc.)]",8.0,"yes, i am not new to coding but am learning ne...",[online courses or certification (includes all...,"yes, i learned how to use ai-enabled tools for...",...,[vertex ai],[],[],[],[chatgpt],[],[when i don't trust ai's answers],"[troubleshooting, profiling, debugging]",61256.0,10.0
2,i am a developer by profession,25-34 years old,29.0,associate degree,employed,[],2.0,"yes, i am not new to coding but am learning ne...","[books / physical media, online courses or cer...","yes, i learned how to use ai-enabled tools for...",...,[],[],[],[],[],[],"[when i don't trust ai's answers, when i have ...",[all skills. ai is a flop.],104413.0,9.0
3,i am a developer by profession,35-44 years old,39.0,bachelor's degree,"independent contractor, freelancer, or self-em...",[none of the above],10.0,"yes, i am not new to coding but am learning ne...",[online courses or certification (includes all...,"yes, i learned how to use ai-enabled tools for...",...,[],[],[],[],"[chatgpt, claude code, github copilot, google ...",[],"[when i don't trust ai's answers, when i have ...","[understand how things actually work, problem ...",53061.0,8.0
4,i am a developer by profession,35-44 years old,39.0,bachelor's degree,employed,[none of the above],4.0,"yes, i am not new to coding but am learning ne...","[ai codegen tools or ai-enabled apps, other on...","yes, i learned how to use ai-enabled tools for...",...,[],[],[],[],"[chatgpt, claude code]",[],"[when i don't trust ai's answers, when i have ...",[],36197.0,6.0
5,i am a developer by profession,35-44 years old,39.0,master's degree,"independent contractor, freelancer, or self-em...","[caring for dependents (children, elderly, etc.)]",21.0,"no, i am not new to coding and did not learn n...",[],"yes, i learned how to use ai-enabled tools for...",...,[],[],[],[],[],[],[when i don't trust ai's answers],"[critical thinking, the skill to define the ta...",60000.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49119,i am a developer by profession,25-34 years old,29.0,bachelor's degree,employed,[],9.0,"yes, i am not new to coding but am learning ne...",[online courses or certification (includes all...,"yes, i learned how to use ai-enabled tools req...",...,[],[],[],[],[],[],[],[],NaN,8.0
49120,i am a developer by profession,35-44 years old,39.0,bachelor's degree,employed,"[caring for dependents (children, elderly, etc.)]",13.0,"no, i am not new to coding and did not learn n...",[],"yes, i learned how to use ai-enabled tools req...",...,[],[],[],[],[],[],[],[],NaN,NaN
49121,i am a developer by profession,25-34 years old,29.0,secondary school,employed,[],2.0,"yes, i am not new to coding but am learning ne...","[blogs or podcasts, books / physical media, ot...","no, i didn't spend time learning in the past year",...,[],[],[],[],[],[],[],[],NaN,NaN


In [32]:
df.to_csv("survey_results_shortened.csv", index=False)
